In [1]:
import os
import random
import cv2
import numpy as np
from pathlib import Path

In [2]:
# --- 1. Path Configuration ---
BASE_DIR = Path("./dataset/Training")
CROP_TRAIN_DIR = Path("./dataset/Croped_train")
CROP_TEST_DIR = Path("./dataset/Croped_test")

CLASSES = ["glioma_tumor", "meningioma_tumor", "no_tumor", "pituitary_tumor"]
SPLIT_RATIO = 0.8  # 80% Train, 20% Test

In [3]:
# --- 2. Create Target Directories ---
def create_class_directories(base_path: Path, classes: list):
    """Create target folders for preprocessed dataset."""
    for cls in classes:
        (base_path / cls).mkdir(parents=True, exist_ok=True)

In [4]:
create_class_directories(CROP_TRAIN_DIR, CLASSES)
create_class_directories(CROP_TEST_DIR, CLASSES)

In [5]:
# --- 3. Crop Image Function ---
def crop_image(image: np.ndarray) -> np.ndarray:
    """Crop brain region by extracting extreme points of the largest contour."""
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    blurred = cv2.GaussianBlur(gray, (5, 5), 0)

    thresh = cv2.threshold(blurred, 45, 255, cv2.THRESH_BINARY)[1]
    thresh = cv2.erode(thresh, None, iterations=2)
    thresh = cv2.dilate(thresh, None, iterations=2)

    contours, _ = cv2.findContours(
        thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE
    )

    if not contours:
        return image  # Return original if no contour found

    c = max(contours, key=cv2.contourArea)

    extLeft = tuple(c[c[:, :, 0].argmin()][0])
    extRight = tuple(c[c[:, :, 0].argmax()][0])
    extTop = tuple(c[c[:, :, 1].argmin()][0])
    extBot = tuple(c[c[:, :, 1].argmax()][0])

    return image[extTop[1] : extBot[1], extLeft[0] : extRight[0]]

In [6]:
# --- 4. Crop, Split, and Save Pipeline ---
def process_split_and_save(
    source_base: Path, train_base: Path, test_base: Path, train_ratio: float
):
    """Loads images directly from class folders, crops them, splits 80/20, and saves them."""
    random.seed(42)  # Set seed for reproducible splits

    for cls in CLASSES:
        class_dir = source_base / cls

        if not class_dir.exists():
            print(f"Directory not found: {class_dir}")
            continue

        # Get all image files directly from BASE_DIR / cls
        class_files = list(class_dir.glob("*.[jJ][pP][gG]")) + list(
            class_dir.glob("*.[pP][nN][gG]")
        )

        if not class_files:
            print(f"No images found in {class_dir}")
            continue

        # Shuffle files randomly before splitting
        random.shuffle(class_files)

        # Calculate index split point
        split_index = int(len(class_files) * train_ratio)
        train_files = class_files[:split_index]
        test_files = class_files[split_index:]

        print(
            f"Processing '{cls}': Total={len(class_files)} | Train={len(train_files)} | Test={len(test_files)}"
        )

        # Process and save Training images
        for img_path in train_files:
            img = cv2.imread(str(img_path))
            if img is not None:
                cropped = crop_image(img)
                cv2.imwrite(str(train_base / cls / img_path.name), cropped)

        # Process and save Testing images
        for img_path in test_files:
            img = cv2.imread(str(img_path))
            if img is not None:
                cropped = crop_image(img)
                cv2.imwrite(str(test_base / cls / img_path.name), cropped)

In [7]:
# ---5. Execute processing ---
process_split_and_save(
    BASE_DIR, CROP_TRAIN_DIR, CROP_TEST_DIR, train_ratio=SPLIT_RATIO
)
print("Done! Images cropped and split into 80% train and 20% test sets.")

Processing 'glioma_tumor': Total=926 | Train=740 | Test=186
Processing 'meningioma_tumor': Total=937 | Train=749 | Test=188
Processing 'no_tumor': Total=492 | Train=393 | Test=99
Processing 'pituitary_tumor': Total=901 | Train=720 | Test=181
Done! Images cropped and split into 80% train and 20% test sets.
